# Priority Product Test Group Creation - Blocked Randomized Design

## Objective

Assign **US-Domestic** OD markets to a controlled pricing experiment using blocked randomization. The allocation ratio and price multipliers are **configurable** via variables in the Configuration cell (`N_CONTROL_PER_BLOCK`, `N_TREAT_PER_ARM`, `FACTOR_MINUS`, `FACTOR_PLUS`).

### Current settings (as of Aug 4, 2026)

| Parameter | Value |
| --- | --- |
| Control / Treatment split | 60% / 20% / 20% (3:1:1 per block) |
| Price multiplier (minus arm) | 0.80 (-20%) |
| Price multiplier (plus arm) | 1.20 (+20%) |
| Test window | Aug 6, 2026 – Dec 5, 2026 |

These are not fixed — change the four design variables to run a different ratio or different price factors in future tests (e.g. 80/10/10 at ±10%).

### Arms

- **Control** — no price change (multiplier 1.00); left unchanged but recorded.
- **T_minus20** — price reduced by the minus factor.
- **T_plus20** — price increased by the plus factor.

The arms are balanced on market-level pre-treatment characteristics jointly:

- Priority Group
- Flight duration
- Day-of-week (DOW) demand profile
- Traveler-mix segment composition (business / bleisure / VFR / vacation / personal)
- PNR volume

## Design

Blocked randomization with a **block size derived from the ratio** (currently 5 = 3 Control + 1 per treatment arm):

1. Build OD-level market features (Domestic only).
2. Log-transform PNR volume (it is highly skewed) and drop pax count (collinear with PNR).
3. Standardize the continuous features, then **whiten** them so distances are Mahalanobis.
4. Form blocks of nearest-neighbor markets, **primarily within exact Priority Group x flight duration cells**.
5. Within each block, randomly assign markets to Control and the two treatment arms per the configured ratio.
6. Apply light rerandomization to select a balanced draw (Mahalanobis across the three arms + a volume-balance penalty **between the two treatment arms**).
7. Validate on OD counts, PG mix, flight-duration mix, DOW profile, traveler mix, PNR/PAX volume, and Mahalanobis distance.

## Notes on inference

Treatment is randomized within blocks, but the final draw is chosen by rerandomization. Downstream analysis should account for the blocked / rerandomized design or use covariate-adjusted modeling.

## Outputs

- `priority_test_group_assignment_blocked` — full OD-to-arm assignment (Control / T_minus20 / T_plus20).
- `priority_pilot_plan_treatment` / `priority_pilot_plan_control` — deployment plans exported as CSVs with pricing-tool header.

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import itertools

RANDOM_SEED = 42

# === Test design parameters (adjust for future tests) ===
# These 4 variables fully define the allocation split and pricing.
# The ratio is N_CONTROL : N_TREAT : N_TREAT per block.
#   e.g. 3:1:1 → block of 5 → 60% / 20% / 20%
#        8:1:1 → block of 10 → 80% / 10% / 10%
#        4:1:1 → block of 6 → ~67% / ~17% / ~17%
N_CONTROL_PER_BLOCK = 3       # How many markets per block stay in Control (no price change)
N_TREAT_PER_ARM = 1           # How many markets per block go to each treatment arm
FACTOR_MINUS = 0.80           # Price multiplier applied to the T_minus arm (e.g. 0.80 = -20%)
FACTOR_PLUS = 1.20            # Price multiplier applied to the T_plus arm (e.g. 1.20 = +20%)

# === Derived arm configuration ===
CONTROL_LABEL = 'Control'
TREATMENT_ARMS = ['T_minus20', 'T_plus20']
ARMS = [CONTROL_LABEL] + TREATMENT_ARMS

BLOCK_SIZE = N_CONTROL_PER_BLOCK + N_TREAT_PER_ARM * len(TREATMENT_ARMS)

TREATMENT_VALUE_MAP = {
    CONTROL_LABEL: 1.00,
    'T_minus20': FACTOR_MINUS,
    'T_plus20': FACTOR_PLUS,
}

# Allocation summary (printed for clarity).
_ctrl_pct = 100 * N_CONTROL_PER_BLOCK / BLOCK_SIZE
_treat_pct = 100 * N_TREAT_PER_ARM / BLOCK_SIZE
print(f'Design: {_ctrl_pct:.0f}% Control / {"/".join(f"{_treat_pct:.0f}%" for _ in TREATMENT_ARMS)} Treatment')
print(f'Block size: {BLOCK_SIZE} | Factors: {FACTOR_MINUS} / {FACTOR_PLUS}')

DOW_COLS = [
    'pct_sun',
    'pct_mon',
    'pct_tue',
    'pct_wed',
    'pct_thu',
    'pct_fri',
    'pct_sat',
]

# OD-level traveler-mix composition (mean of the per-row segment probabilities).
# Included because traveler segment drives price elasticity, which the DOW
# profile only weakly proxies.
SEG_COLS = [
    'seg_business',
    'seg_bleisure',
    'seg_vfr',
    'seg_vacation',
    'seg_personal',
]

START_DATE = '2026-08-06'
END_DATE = '2026-12-05'

In [0]:
# # Establish the Databricks Connect session for this (fresh) kernel.
# # getOrCreate() with no prior spark.stop() is safe here - it just attaches.
# from databricks.connect import DatabricksSession

# spark = DatabricksSession.builder.getOrCreate()
# print('Spark session ready. Test:', spark.range(3).count())

In [0]:
priority_airport = pd.read_excel('Priority_Group_Airport.xlsx')

priority_airport_spark = spark.createDataFrame(priority_airport)
priority_airport_spark.createOrReplaceTempView('priority_airport')

finalTransactionOfferSale = spark.table('rm_workspace.finalTransactionOfferSale_B')
finalTransactionOfferSale.createOrReplaceTempView('finalTransactionOfferSale')

print(f'finalTransactionOfferSale_B: {finalTransactionOfferSale.count():,} rows')

In [0]:
itinerary = (
    spark.table('rm_workspace.tmp_pnr_spine')
    .filter(F.length(F.col('fare_basis_cd')) <= 8)
    .filter(F.col('pax_count') > 0)
)

itinerary.createOrReplaceTempView('itinerary')

In [0]:
priority_airport_sdf = spark.table('priority_airport')
finalTransactionOfferSale = spark.table('finalTransactionOfferSale')

offer_sale = (
    finalTransactionOfferSale.alias('f')
    .join(
        priority_airport_sdf.alias('p'),
        F.col('f.od_origin') == F.col('p.Airport'),
        'left',
    )
    .withColumn('DOW', F.dayofweek('OD_dep_dt'))
    .withColumn('Priority_Group', F.coalesce(F.col('p.Group'), F.lit(6)))
)

offer_sale.createOrReplaceTempView('offer_sale')

In [0]:
dow_counts = (
    itinerary
    .filter(F.col('region').like('%US48%'))
    .groupBy(
        F.col('od_origin_airprt_iata_cd').alias('od_origin'),
        F.col('od_destntn_airprt_iata_cd').alias('od_destination'),
        F.dayofweek('od_local_dep_dt').alias('dow'),
    )
    .agg(F.count('*').alias('dep_cnt'))
)

totals = (
    dow_counts
    .groupBy('od_origin', 'od_destination')
    .agg(F.sum('dep_cnt').alias('total_dep'))
)

od_dow = (
    dow_counts.alias('d')
    .join(totals.alias('t'), ['od_origin', 'od_destination'])
    .groupBy('od_origin', 'od_destination')
    .agg(
        F.round(100 * F.sum(F.when(F.col('dow') == 1, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_sun'),
        F.round(100 * F.sum(F.when(F.col('dow') == 2, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_mon'),
        F.round(100 * F.sum(F.when(F.col('dow') == 3, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_tue'),
        F.round(100 * F.sum(F.when(F.col('dow') == 4, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_wed'),
        F.round(100 * F.sum(F.when(F.col('dow') == 5, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_thu'),
        F.round(100 * F.sum(F.when(F.col('dow') == 6, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_fri'),
        F.round(100 * F.sum(F.when(F.col('dow') == 7, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_sat'),
    )
)

od_dow.createOrReplaceTempView('od_dow')

In [0]:
pnr_od = (
    itinerary
    .filter(F.col('region').like('%US48%'))
    .groupBy(
        F.col('od_origin_airprt_iata_cd').alias('od_origin'),
        F.col('od_destntn_airprt_iata_cd').alias('od_destination'),
        'pnr_loctr_id',
    )
    .agg(F.max('pax_count').alias('pax_count'))
)

od_volume = (
    pnr_od
    .groupBy('od_origin', 'od_destination')
    .agg(
        F.countDistinct('pnr_loctr_id').alias('pnr_cnt'),
        F.sum('pax_count').alias('pax_cnt'),
    )
)

od_volume.createOrReplaceTempView('od_volume')

In [0]:
od_market_features = (
    offer_sale.alias('o')
    .join(od_dow.alias('d'), ['od_origin', 'od_destination'], 'left')
    .join(od_volume.alias('v'), ['od_origin', 'od_destination'], 'left')
    .filter(F.col('region_group') == 'Domestic')
    .groupBy(
        'od_origin',
        'od_destination',
        'pct_sun',
        'pct_mon',
        'pct_tue',
        'pct_wed',
        'pct_thu',
        'pct_fri',
        'pct_sat',
        'pnr_cnt',
        'pax_cnt',
    )
    .agg(
        F.max('Priority_Group').alias('priority_group'),
        F.max('FlightDuration').alias('flight_duration'),
        F.avg('avg_business_prob').alias('seg_business'),
        F.avg('avg_bleisure_prob').alias('seg_bleisure'),
        F.avg('avg_vfr_prob').alias('seg_vfr'),
        F.avg('avg_vacation_prob').alias('seg_vacation'),
        F.avg('avg_personal_prob').alias('seg_personal'),
    )
    .withColumn('pnr_cnt', F.coalesce(F.col('pnr_cnt'), F.lit(0)))
    .withColumn('pax_cnt', F.coalesce(F.col('pax_cnt'), F.lit(0)))
)

od_market_features.createOrReplaceTempView('od_market_features')

In [0]:
od_features = spark.sql('''
SELECT
    od_origin,
    od_destination,
    priority_group,
    flight_duration,
    pct_sun,
    pct_mon,
    pct_tue,
    pct_wed,
    pct_thu,
    pct_fri,
    pct_sat,
    pnr_cnt,
    pax_cnt,
    seg_business,
    seg_bleisure,
    seg_vfr,
    seg_vacation,
    seg_personal
FROM od_market_features
''').toPandas()

for c in DOW_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce')

for c in SEG_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce')

for c in ['pnr_cnt', 'pax_cnt']:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce').fillna(0)

od_features[DOW_COLS] = od_features[DOW_COLS].fillna(0)
od_features[SEG_COLS] = od_features[SEG_COLS].fillna(0)

print(f'OD markets: {len(od_features):,}')
display(od_features.head())

## Feature preparation

Build the feature vector used for blocking and balance scoring. Volume is log-transformed because it is heavy-tailed; pax count is dropped as it is collinear with PNR count. The DOW demand profile and the traveler-mix segment shares (business / bleisure / VFR / vacation / personal) are added as continuous features. All standardized features are then **whitened**, so Euclidean distance on the whitened coordinates equals Mahalanobis distance - this keeps the correlated DOW and segment columns from dominating volume and drops their exact sum dependencies. Priority Group and flight duration are kept as exact-match keys rather than distances.

In [0]:
# Continuous balancing features, standardized to a common scale.
# Volume is heavy-tailed, so use log1p. pax_cnt is dropped because it is
# almost perfectly collinear with pnr_cnt and would double-count volume.
# The DOW shares and the traveler-mix segment shares are each compositional
# (each set sums to a constant), so one reference column is dropped from each
# to remove the exact linear dependency. No information is lost - the dropped
# share is implied by the others - and it keeps the whitening well-posed
# (otherwise rounding noise in the near-null "sum" direction gets amplified).
od_features['log_pnr'] = np.log1p(od_features['pnr_cnt'])

DOW_BLOCK_COLS = DOW_COLS[:-1]   # drop pct_sat (implied by the other six)
SEG_BLOCK_COLS = SEG_COLS[:-1]   # drop seg_personal (implied by the other four)

CONT_FEATURES = ['log_pnr'] + DOW_BLOCK_COLS + SEG_BLOCK_COLS
Z_COLS = ['z_' + c for c in CONT_FEATURES]

mu = od_features[CONT_FEATURES].mean()
sd = od_features[CONT_FEATURES].std(ddof=0).replace(0, 1.0)
od_features[Z_COLS] = (od_features[CONT_FEATURES] - mu) / sd

# Whitening transform: Euclidean distance on the whitened coordinates equals
# Mahalanobis distance in the standardized space. This fixes the feature-
# weighting problem - without it the DOW and segment columns would swamp the
# single volume dimension. With the reference columns dropped above, the
# covariance is full rank, so no eigen-direction should need zeroing.
Zc = od_features[Z_COLS].to_numpy(dtype=float)
cov_z = np.cov(Zc, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov_z)
tol = 1e-8 * eigvals.max()
inv_sqrt = np.where(eigvals > tol, 1.0 / np.sqrt(eigvals), 0.0)
WHITEN = eigvecs @ np.diag(inv_sqrt) @ eigvecs.T
W_COLS = ['w_' + c for c in CONT_FEATURES]
od_features[W_COLS] = Zc @ WHITEN

# Categorical design factors are enforced as exact-match block keys, so every
# block is homogeneous on Priority Group and flight duration.
od_features['block_key'] = (
    od_features['priority_group'].astype(str)
    + '|'
    + od_features['flight_duration'].astype(str)
)

n_keys = od_features['block_key'].nunique()
print(f'Continuous features (standardized): {CONT_FEATURES}')
print(f'Effective whitened dimensions: {int((inv_sqrt > 0).sum())} of {len(CONT_FEATURES)}')
print(f'Exact-match keys (PG x duration): {n_keys}')

## Blocking

### What is a block?

A **block** is a small group of markets that are as similar as possible on observable characteristics.

**Why is the block size 5?** Because each block is a miniature version of the full experiment — it must contain exactly the right number of markets to fill every arm at the target ratio. A 60/20/20 split simplifies to **3:1:1** (the smallest whole-number representation), so you need 3 Control + 1 T_minus + 1 T_plus = **5 markets per block**. If you changed the design to 80/10/10, the ratio becomes 8:1:1, so blocks would be 10. The formula is `N_CONTROL_PER_BLOCK + N_TREAT_PER_ARM × number_of_treatment_arms`.

The idea is simple: if the markets within a block look nearly identical before the test, any difference in outcomes after the test can be attributed to the price change rather than pre-existing market differences.

### How blocks are formed

1. **Exact-match constraint first** — markets are only grouped together if they share the same **Priority Group** and **flight duration**. This guarantees within-block homogeneity on the two most important categorical dimensions.
2. **Nearest-neighbor within those cells** — among markets that share the same PG × duration, we find the 5 closest neighbors using Mahalanobis distance (accounting for DOW profile, traveler mix, and volume simultaneously).
3. **Remainder pass** — markets that can't form a full block within their exact-match cell are pooled and blocked without the exact-match constraint (still using nearest-neighbor distance).
4. **Leftover** — any markets that can't even form a block in the remainder pass are assigned directly to Control.

### How assignment works within each block

Once blocks are formed, each block is split according to the configured ratio (`N_CONTROL_PER_BLOCK` markets stay as Control, `N_TREAT_PER_ARM` markets go to each treatment arm). Because every market in a block is nearly identical on pre-treatment features, each treated market has closely matched control markets right next to it — enabling clean within-block comparisons of the price effect.

In [0]:
Wmat = od_features[W_COLS]


def form_blocks(features_df, dist_mat, block_key_col, k, seed):
    # Greedy nearest-neighbor blocking into blocks of size k, using Euclidean
    # distance on the whitened coordinates (equivalent to Mahalanobis).
    # Pass 1 blocks within each exact-match key; markets that cannot fill a full
    # block go to a remainder pool. Pass 2 blocks the remainder ignoring the
    # exact-match constraint. Any final leftover (< k) keeps a NaN block id.
    # Implemented on NumPy arrays with an alive-mask (no per-row pandas .loc),
    # so it scales to tens of thousands of markets in seconds.
    rng = np.random.default_rng(seed)
    Z = dist_mat.to_numpy(dtype=float)
    keys = features_df[block_key_col].to_numpy()
    n = len(features_df)
    block_of_pos = np.full(n, -1, dtype=np.int64)
    counter = {'next': 0}

    def greedy(pos_arr):
        # pos_arr holds global row positions. Form blocks of the k nearest
        # still-alive markets around a rolling seed. Returns leftover positions.
        pos = np.asarray(pos_arr, dtype=np.int64)
        rng.shuffle(pos)
        Zsub = Z[pos]
        alive = np.ones(len(pos), dtype=bool)
        ptr = 0
        n_alive = len(pos)
        while n_alive >= k:
            while not alive[ptr]:
                ptr += 1
            alive_idx = np.flatnonzero(alive)
            d = np.sqrt(((Zsub[alive_idx] - Zsub[ptr]) ** 2).sum(axis=1))
            nearest = alive_idx[np.argsort(d)[:k]]
            block_of_pos[pos[nearest]] = counter['next']
            counter['next'] += 1
            alive[nearest] = False
            n_alive -= k
        return pos[alive].tolist()

    remainder = []
    for key in pd.unique(keys):
        remainder.extend(greedy(np.flatnonzero(keys == key)))

    final_leftover_pos = greedy(remainder)

    block_id = pd.Series(
        np.where(block_of_pos >= 0, block_of_pos, np.nan),
        index=features_df.index,
    )
    final_leftover = features_df.index[final_leftover_pos].tolist()
    return block_id, final_leftover


block_id, leftover = form_blocks(od_features, Wmat, 'block_key', BLOCK_SIZE, RANDOM_SEED)
od_features['block_id'] = block_id

n_blocks = int(od_features['block_id'].notna().sum() // BLOCK_SIZE)
print(f'Full blocks of {BLOCK_SIZE} formed: {n_blocks} (covering {n_blocks * BLOCK_SIZE:,} markets)')
print(f'Treated markets: {n_blocks * len(TREATMENT_ARMS):,} '
      f'({100 * n_blocks * len(TREATMENT_ARMS) / len(od_features):.1f}% of all markets)')
print(f'Leftover markets (assigned to Control): {len(leftover)}')

# Diagnostic: how many blocks mix Priority Group x duration keys (these can only
# arise in the remainder pass, which drops the exact-match constraint). If this
# is tiny, the exact-match design effectively holds for all markets.
blocked = od_features.dropna(subset=['block_id'])
keys_per_block = blocked.groupby('block_id')['block_key'].nunique()
mixed_blocks = keys_per_block[keys_per_block > 1]
markets_in_mixed = int(od_features['block_id'].isin(mixed_blocks.index).sum())
print(f'Blocks mixing PG x duration keys: {len(mixed_blocks):,}')
print(f'Markets in mixed blocks: {markets_in_mixed:,} '
      f'({100 * markets_in_mixed / len(od_features):.1f}% of all markets)')

## Balance metric

### How do we know the groups are balanced?

After assigning markets to arms, we check whether the three groups (Control, T_minus20, T_plus20) "look the same" on average across all features. The metric used is the **Mahalanobis distance** between group means — this is a single number that summarizes how far apart the group centroids are, accounting for correlations between features.

- **Lower = better balanced.** A score near 0 means the groups are virtually indistinguishable on pre-treatment characteristics.
- The metric uses the same distance geometry as the blocking step, so design and evaluation are consistent.
- The inverse covariance matrix (`VI_GLOBAL`) is computed once (it doesn't depend on arm labels) and reused across all rerandomization trials.

In [0]:
def compute_VI(df, feature_cols):
    # Inverse covariance (pseudo-inverse) of the market-level features. This
    # does not depend on arm labels, so it is computed once and reused.
    X = df[feature_cols].to_numpy(dtype=float)
    cov = np.cov(X, rowvar=False)
    return np.linalg.pinv(cov)


def arm_balance_mahalanobis(df, feature_cols, VI=None, group_col='test_group'):
    # Overall imbalance = mean pairwise Mahalanobis distance between arm
    # mean-vectors. Scale-free and correlation-aware, so mixed units and
    # collinear features are handled automatically. Lower is better.
    # Pass a precomputed VI to avoid recomputing the covariance every call.
    if VI is None:
        VI = compute_VI(df, feature_cols)
    means = df.groupby(group_col)[feature_cols].mean()
    arms = list(means.index)
    dists = {}
    for a, b in itertools.combinations(arms, 2):
        diff = (means.loc[a] - means.loc[b]).to_numpy()
        dists[a + ' vs ' + b] = float(np.sqrt(diff @ VI @ diff))
    overall = float(np.mean(list(dists.values())))
    return overall, pd.Series(dists)


# Covariance is invariant to arm labels, so precompute the inverse once.
VI_GLOBAL = compute_VI(od_features, Z_COLS)

## Assignment and rerandomization

### Why not just randomize once?

A single random assignment might, by chance, put all high-volume markets into one arm. **Rerandomization** fixes this: we draw many random assignments (2,000 by default), score each one, and keep the best.

### How it works

1. Within each block (size = `N_CONTROL_PER_BLOCK + N_TREAT_PER_ARM × 2`), randomly pick `N_TREAT_PER_ARM` markets for each treatment arm and assign the rest to Control.
2. Repeat this random draw 2,000 times with different seeds.
3. Score each draw on a combined objective:
   - **Mahalanobis imbalance** — how different the three arm means are across all features (lower = more balanced).
   - **Treatment-arm volume gap** — the PNR volume difference between T_minus20 and T_plus20 (we want the two treatment arms to carry similar traffic so the price effect is comparable). Control is ~3× each treatment arm by design, so only the two treatment arms are compared on total volume.
4. Keep the draw with the lowest combined score.

Leftover markets (fewer than a full block) are assigned to Control regardless.

In [0]:
# Vectorized rerandomization for the 60 / 20 / 20 design.
# Codes: 0 = Control, 1 = T_minus20, 2 = T_plus20.
# Within each block of BLOCK_SIZE, exactly one market goes to each treatment arm
# and the remaining BLOCK_SIZE - 2 go to Control. Leftover markets (that never
# formed a full block) stay Control.
Zvals = od_features[Z_COLS].to_numpy(dtype=float)
pnr = od_features['pnr_cnt'].to_numpy(dtype=float)
n_rows = len(od_features)
positions = np.arange(n_rows)

full_mask = od_features['block_id'].notna().to_numpy()
full_pos = positions[full_mask]
full_blk = od_features['block_id'].to_numpy()[full_mask]

# Sort full markets by block id so each consecutive BLOCK_SIZE positions is one
# block (every full block has exactly BLOCK_SIZE members by construction).
order = np.argsort(full_blk, kind='stable')
block_member_pos = full_pos[order].reshape(-1, BLOCK_SIZE)
leftover_pos = positions[~full_mask]

n_blocks_full = block_member_pos.shape[0]
block_rows = np.arange(n_blocks_full)

N_GROUPS = len(ARMS)   # 3: Control, T_minus20, T_plus20
pair_a, pair_b = zip(*itertools.combinations(range(N_GROUPS), 2))
pair_a = np.array(pair_a)
pair_b = np.array(pair_b)

# Weight on treatment-arm volume balance vs the Mahalanobis score.
VOLUME_WEIGHT = 1.0


def build_codes(seed):
    # Everything Control (0), then place one -20% (1) and one +20% (2) per block.
    rng = np.random.default_rng(seed)
    codes = np.zeros(n_rows, dtype=np.int64)
    picks = np.argsort(rng.random((n_blocks_full, BLOCK_SIZE)), axis=1)[:, :2]
    codes[block_member_pos[block_rows, picks[:, 0]]] = 1   # T_minus20
    codes[block_member_pos[block_rows, picks[:, 1]]] = 2   # T_plus20
    return codes


def score_components(codes):
    # Mean pairwise Mahalanobis distance across the three arm mean-vectors.
    means = np.vstack([Zvals[codes == a].mean(axis=0) for a in range(N_GROUPS)])
    diffs = means[pair_a] - means[pair_b]
    maha = np.sqrt(np.einsum('ij,jk,ik->i', diffs, VI_GLOBAL, diffs)).mean()
    # Volume balance BETWEEN the two treatment arms (control is ~3x by design).
    pnr_minus = pnr[codes == 1].sum()
    pnr_plus = pnr[codes == 2].sum()
    vol_gap = abs(pnr_minus - pnr_plus) / ((pnr_minus + pnr_plus) / 2)
    return maha, vol_gap


def balance_score(codes):
    maha, vol_gap = score_components(codes)
    return maha + VOLUME_WEIGHT * vol_gap


R = 2000
best_score = np.inf
best_codes = None
best_trial = None

for r in range(R):
    codes = build_codes(RANDOM_SEED + r)
    s = balance_score(codes)
    if s < best_score:
        best_score = s
        best_codes = codes
        best_trial = r

label_arr = np.array(ARMS, dtype=object)   # 0->Control, 1->T_minus20, 2->T_plus20
od_features['test_group'] = label_arr[best_codes]

best_maha, best_vol_gap = score_components(best_codes)
counts = od_features['test_group'].value_counts()
print(f'Best combined score: {best_score:.4f} (trial {best_trial} of {R})')
print(f'  Mahalanobis imbalance (3 arms): {best_maha:.4f}')
print(f'  Treatment-arm PNR gap:          {100 * best_vol_gap:.2f}% of treatment mean')
print('  Arm sizes: ' + ', '.join(f'{a}={int(counts.get(a, 0)):,}' for a in ARMS))

In [0]:
def validate_assignment(df, title='BALANCE SUMMARY'):
    print('=' * 70)
    print(title)
    print('=' * 70)

    count_balance = df.groupby('test_group').size()
    print('\nOD count by group (60 / 20 / 20 by design):')
    print(count_balance)

    pg_balance = pd.crosstab(df['test_group'], df['priority_group'], normalize='index') * 100
    print('\nPriority Group mix:')
    display(pg_balance.round(2))
    print('\nPriority Group max-min percentage point imbalance:')
    print((pg_balance.max() - pg_balance.min()).round(3))

    fd_balance = pd.crosstab(df['test_group'], df['flight_duration'], normalize='index') * 100
    print('\nFlight Duration mix:')
    display(fd_balance.round(2))
    print('\nFlight Duration max-min percentage point imbalance:')
    print((fd_balance.max() - fd_balance.min()).round(3))

    dow_balance = df.groupby('test_group')[DOW_COLS].mean()
    print('\nDOW profile:')
    display(dow_balance.round(2))
    print('\nDOW max-min percentage point imbalance:')
    print((dow_balance.max() - dow_balance.min()).round(3))

    seg_balance = df.groupby('test_group')[SEG_COLS].mean()
    print('\nTraveler-mix segment profile (mean probability):')
    display(seg_balance.round(3))
    print('\nTraveler-mix max-min imbalance:')
    print((seg_balance.max() - seg_balance.min()).round(4))

    volume_summary = (
        df.groupby('test_group')
        .agg(
            n_ods=('od_origin', 'size'),
            total_pnrs=('pnr_cnt', 'sum'),
            avg_pnrs_per_od=('pnr_cnt', 'mean'),
            total_pax=('pax_cnt', 'sum'),
            avg_pax_per_od=('pax_cnt', 'mean'),
        )
        .round(2)
    )
    print('\nVolume summary:')
    display(volume_summary)

    # Total volume per arm is ~3:1:1 by design; the balance that matters is
    # between the two treatment arms, plus per-market averages across all arms.
    treat = [a for a in TREATMENT_ARMS if a in volume_summary.index]
    if len(treat) == 2:
        tmin = volume_summary.loc[treat[0], 'total_pnrs']
        tplus = volume_summary.loc[treat[1], 'total_pnrs']
        gap = abs(tmin - tplus)
        gap_pct = 100 * gap / ((tmin + tplus) / 2)
        print(f'\nTreatment-arm PNR gap ({treat[0]} vs {treat[1]}):')
        print(f'{gap:,.0f} PNRs ({gap_pct:.2f}% of treatment mean)')

    print('\nAvg PNRs per OD (should match across arms if treated is representative):')
    print(volume_summary['avg_pnrs_per_od'])

    return {
        'count_balance': count_balance,
        'pg_balance': pg_balance,
        'fd_balance': fd_balance,
        'dow_balance': dow_balance,
        'seg_balance': seg_balance,
        'volume_summary': volume_summary,
    }


blocked_validation = validate_assignment(od_features, title='BLOCKED ASSIGNMENT BALANCE SUMMARY')

overall, pair_dists = arm_balance_mahalanobis(od_features, Z_COLS, VI=VI_GLOBAL)
print('\nMahalanobis imbalance (overall mean pairwise):', round(overall, 4))
print('\nPairwise Mahalanobis distances:')
display(pair_dists.round(4))

print('\nPer-arm standardized feature means (closer to each other = better):')
display(od_features.groupby('test_group')[Z_COLS].mean().round(3))

## Output

Register the blocked assignment as a temp view. The table write is left commented out, mirroring the original notebook.

In [0]:
final_assignment = od_features[
    [
        'od_origin',
        'od_destination',
        'priority_group',
        'flight_duration',
        'test_group',
        'pnr_cnt',
        'pax_cnt',
    ]
].copy()

final_assignment_spark = spark.createDataFrame(final_assignment)
final_assignment_spark.createOrReplaceTempView('priority_test_group_assignment_blocked')

# final_assignment_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_test_group_assignment_blocked'
# )

## Deployment plan

Build a single **combined** plan over all markets. Treated markets carry their price multiplier in `Result`; **Control markets are kept for recording**, clearly flagged via the `arm` column and `Comment`, with `Result` = 1.00. Each row is formatted in the pricing-tool layout (a `Loc1` origin/destination pair and travel-date window). Registered as `priority_pilot_plan_blocked`.

In [0]:
# All markets active for the full test window.
plan = od_features.copy()
plan['Travel dates (Start)'] = START_DATE
plan['Travel dates (End)'] = END_DATE

plan['Loc1'] = 'P:' + plan['od_origin'].astype(str)
plan['Loc2'] = 'P:' + plan['od_destination'].astype(str)
plan['Sector direction'] = 'From Loc1 to Loc2'

# Clearly differentiate Control (recorded, no change) from the treated arms.
plan['arm'] = plan['test_group']
plan['Comment'] = np.where(
    plan['test_group'] == CONTROL_LABEL,
    'Control - no change',
    'Random Testing',
)
plan['Segment matches required'] = ''
plan['Result'] = plan['test_group'].map(TREATMENT_VALUE_MAP)

priority_plan = plan[
    [
        'arm',
        'Comment',
        'Segment matches required',
        'Travel dates (Start)',
        'Travel dates (End)',
        'Loc1',
        'Loc2',
        'Sector direction',
        'Result',
    ]
].copy()

priority_plan['Travel dates (Start)'] = pd.to_datetime(priority_plan['Travel dates (Start)']).dt.strftime('%m/%d/%Y')
priority_plan['Travel dates (End)'] = pd.to_datetime(priority_plan['Travel dates (End)']).dt.strftime('%m/%d/%Y')

# Split into separate treatment and control exports.
priority_plan_treatment = priority_plan[priority_plan['arm'] != CONTROL_LABEL].drop(columns=['arm']).copy()
priority_plan_control = priority_plan[priority_plan['arm'] == CONTROL_LABEL].copy()

print(f'Treatment rows: {len(priority_plan_treatment):,}')
display(priority_plan_treatment.head(10))
print(f'\nControl rows: {len(priority_plan_control):,}')
display(priority_plan_control.head(5))

In [0]:
# Separate temp views for treatment and control.
treatment_spark = spark.createDataFrame(priority_plan_treatment)
treatment_spark.createOrReplaceTempView('priority_pilot_plan_treatment')

control_spark = spark.createDataFrame(priority_plan_control)
control_spark.createOrReplaceTempView('priority_pilot_plan_control')

# treatment_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_pilot_plan_treatment'
# )
# control_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_pilot_plan_control'
# )

In [0]:
priority_template = pd.read_csv("Priority_Test_Template.csv")
priority_template


In [0]:
# Export treatment and control plans to CSV (workspace folder).
# Prepend pricing-tool metadata header rows to match the template structure.
output_dir = '/Workspace/Users/939510@corpaa.aa.com/KRATOS_Exploration'

PRICING_HEADER_ROWS = [
    'Pricing Table',
    'package,Priority',
    'carrier,AA',
    'subcode,CUU',
    'result_type,UNITLESS_PTABLE_TYPE',
    'name,Priority Price Test_t',
    'comment,Updating stale',
    'attribute_group,TS',
    'Pricelines',
]


def export_with_header(df, filepath, header_rows):
    """Write a CSV with pricing-tool metadata rows above the column header."""
    n_cols = len(df.columns)
    with open(filepath, 'w', newline='') as f:
        for row in header_rows:
            # Pad each metadata row with empty columns to match data width.
            n_existing = row.count(',') + 1
            padded = row + ',' * (n_cols - n_existing)
            f.write(padded + '\n')
        df.to_csv(f, index=False)


export_with_header(
    priority_plan_treatment,
    f'{output_dir}/priority_pilot_plan_treatment.csv',
    PRICING_HEADER_ROWS,
)
export_with_header(
    priority_plan_control,
    f'{output_dir}/priority_pilot_plan_control.csv',
    PRICING_HEADER_ROWS,
)

print(f'Treatment exported: {len(priority_plan_treatment):,} rows')
print(f'Control exported:   {len(priority_plan_control):,} rows')
print(f'\nFiles saved to: {output_dir}/')
print(f'Header rows prepended: {len(PRICING_HEADER_ROWS)}')

In [0]:
# Illustration: show a few complete blocks. Each block = 5 look-alike markets
# that share the same Priority Group x flight duration, one handed to each arm.
show_cols = [
    'block_id', 'test_group', 'od_origin', 'od_destination',
    'priority_group', 'flight_duration',
    'pnr_cnt', 'pct_mon', 'pct_sat', 'seg_business', 'seg_vacation',
]

pure = od_features.dropna(subset=['block_id'])
example_ids = pure['block_id'].drop_duplicates().head(3).tolist()

for bid in example_ids:
    blk = pure[pure['block_id'] == bid].sort_values('test_group')
    print(f'--- BLOCK {int(bid)} '
          f'(key = {blk["block_key"].iloc[0]}) ---')
    display(blk[show_cols])

In [0]:
# DOW examples vs trip intent (traveler segment).
# Restrict to reasonably busy markets so the DOW profile is not just noise from
# a handful of departures.
ex = od_features[od_features['pnr_cnt'] >= 500].copy()
ex['weekday_share'] = ex[['pct_mon', 'pct_tue', 'pct_wed', 'pct_thu']].sum(axis=1)
ex['weekend_share'] = ex[['pct_fri', 'pct_sat', 'pct_sun']].sum(axis=1)

cols = [
    'od_origin', 'od_destination', 'pnr_cnt',
    'pct_sun', 'pct_mon', 'pct_tue', 'pct_wed', 'pct_thu', 'pct_fri', 'pct_sat',
    'weekday_share', 'weekend_share',
    'seg_business', 'seg_vacation', 'seg_vfr',
]

print('=== MOST WEEKDAY-HEAVY markets (expect business-leaning trip intent) ===')
display(ex.sort_values('weekday_share', ascending=False).head(6)[cols].round(2))

print('=== MOST WEEKEND-HEAVY markets (expect leisure/vacation-leaning) ===')
display(ex.sort_values('weekend_share', ascending=False).head(6)[cols].round(2))